# Experiment 05: Cloud Vector Store Integration with Pinecone & LCEL
- **Goal:** Demonstrate the flexibility of LangChain by migrating the vector database from a local Chroma DB to a cloud-based **Pinecone** index with minimal configuration changes.
- **Components:** `PineconeVectorStore`, `OpenAIEmbeddings`, `RecursiveCharacterTextSplitter`, `LCEL`.
- **Method:**
    1. Load and chunk documents using the established preprocessing pipeline.
    2. Initialize a Pinecone index and upload document vectors via `PineconeVectorStore.from_documents`.
    3. Re-configure the LCEL chain by swapping the retriever component to point to the Pinecone index.
- **Key Observation:** LangChain's standardized `VectorStore` interface allows for seamless database migration. This experiment highlights the "plug-and-play" capability of the retrieval layer, enabling easy scaling from local development to cloud production environments.

In [ ]:
# pip install docx2txt==0.9 langchain-community==0.4.1 langchain-text-splitters==1.1.2

In [1]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader('./tax.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

In [ ]:
# pip install -qU langchain==1.2.15 langchain-pinecone langchain-openai

In [2]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

embedding = OpenAIEmbeddings(model='text-embedding-3-large')

In [3]:
import os
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

pinecone_api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=pinecone_api_key)

index_name = "tax-index"
docsearch = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name)

In [4]:
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'
retrieved_docs = docsearch.similarity_search(query, k=3)

In [5]:
retrieved_docs

[Document(id='0d064a1c-7f5d-47da-863a-cab4ea5b96c2', metadata={'source': './tax.docx'}, page_content='제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n| 종합소득 과세표준          | 세율                                         |\n\n|-------------------|--------------------------------------------|\n\n| 1,400만원 이하    | 과세표준의 6퍼센트                             |\n\n| 1,400만원 초과   5,000만원 이하    | 84만원 + (1,400만원을 초과하는 금액의 15퍼센트)  |\n\n| 5,000만원 초과   8,800만원 이하    | 624만원 + (5,000만원을 초과하는 금액의 24퍼센트) |\n\n| 8,800만원 초과   1억5천만원 이하    | 1,536만원 + (8,800만원을 초과하는 금액의 35퍼센트)|\n\n| 1억5천만원 초과   3억원 이하    | 3,706만원 + (1억5천만원을 초과하는 금액의 38퍼센트)|\n\n| 3억원 초과   5억원 이하    | 9,406만원 + (3억원을 초과하는 금액의 40퍼센트)   |\n\n| 5억원 초과   10억원 이하    | 1억7,406만원 + (5억원을 초과하는 금액의 42퍼센트)|\n\n| 10억원 초과    | 3억8,406만원 + (10억원을 초과하는 금액의 45퍼센트)|\n\n② 거주자의 퇴직소득에 대한 소득세는 다음 각 호의 순서에 따라 계산한 금액(이하 “퇴직소득 산출세액”이라 한다)으

In [6]:
from langchain_openai import ChatOpenAI
from langsmith import Client

llm = ChatOpenAI(model='gpt-4o')

client = Client()
prompt = client.pull_prompt("rlm/rag-prompt")

In [7]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

retriever = docsearch.as_retriever()

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [8]:
response = rag_chain.invoke(query)

In [9]:
response

'연봉 5천만원인 직장인의 소득세는 624만원입니다. 이는 과세표준 5천만원에 해당되며, 해당 금액을 초과하지 않으므로 84만원 + (1,400만원 초과분 3600만원의 15퍼센트)로 계산됩니다.'